# Multi-Condition ControlNet Stacking: Reproduction Notebook

This notebook demonstrates the fixed-A non-normalized B-weight retention slope measurement pipeline.
Run the core statistical analysis without GPU, then optionally run full generation.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.statistics import compute_slope_summary, ols_slope
from src.lookup_tool import DefaultLookupTool
from src.visualization import plot_slope_summary_panel

print('Imports successful')

## 2. Demonstrate the Lookup Tool

In [ ]:
tool = DefaultLookupTool()

# The only cell with a prespecified heuristic default
print(tool.lookup('canny_depth', 7.5, 'sdxl').message)
print()

# All other cells return 'no recommendation'
print(tool.lookup('canny_depth', 15.0, 'sdxl').message)

## 3. Simulate Per-Prompt Slope Calculation

In [ ]:
# Simulate data matching paper: Canny->Depth s̄ = -0.38
b_weights = [0.1, 0.3, 0.5, 0.7, 0.9]
n_prompts = 20
np.random.seed(42)

x = np.array(b_weights)
retention = np.zeros((n_prompts, 5))
for i in range(n_prompts):
    true_slope = -0.38 + np.random.normal(0, 0.12)
    retention[i] = true_slope * x + 0.85 + np.random.normal(0, 0.02, 5)

summary = compute_slope_summary(retention, b_weights, 'canny->depth on depth_ssim')
print(f's̄ = {summary.mean_signed_slope:.3f}')
print(f'SII = {summary.sii:.3f}')
print(f'MAS = {summary.mas:.3f}')
print(f'95% CI: [{summary.ci_lower:.3f}, {summary.ci_upper:.3f}]')